In [11]:
#STEP 0Setup & load data (run once)

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Load processed EPC data
df = pd.read_csv(
    "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_processed.csv"
)

# Define outcome variable (PEI)
target = "ENERGY_CONSUMPTION_CURRENT"

# Drop rows with missing PEI
df = df.dropna(subset=[target])

df.shape

(136, 118)

In [12]:
# ==========================
# Continuous variables correlation with ENERGY_CONSUMPTION_CURRENT
# ==========================

from scipy.stats import pearsonr, spearmanr

# List of continuous variables to test
continuous_vars = [
    "TOTAL_FLOOR_AREA",
    "3_YR_ENERGY_COST_CURRENT",
    "3_YR_ENERGY_SAVINGS_POTENTIAL",
    "CO2_EMISS_CURR_PER_FLOOR_AREA",
    "SPACE_HEATING_DEMAND",
    "WATER_HEATING_DEMAND",
    "GLAZED_AREA",
    "NUMBER_HABITABLE_ROOMS",
    "NUMBER_HEATED_ROOMS",
    "HEATING_COST_CURRENT",
    "HEATING_COST_POTENTIAL",
    "HOT_WATER_COST_CURRENT",
    "HOT_WATER_COST_POTENTIAL",
    "LIGHTING_COST_CURRENT",
    "LIGHTING_COST_POTENTIAL",
    "EXTENSION_COUNT",
    "LOW_ENERGY_FIXED_LIGHT_COUNT"
]

results_continuous = []

for col in continuous_vars:
    if col in df.columns:
        # Drop missing values for this column
        temp = df[[col, target]].dropna()
        # Skip if column is constant
        if temp[col].nunique() > 1:
            pearson_r, pearson_p = pearsonr(temp[col], temp[target])
            spearman_r, spearman_p = spearmanr(temp[col], temp[target])
            results_continuous.append({
                "Variable": col,
                "Pearson_r": pearson_r,
                "Pearson_p": pearson_p,
                "Spearman_r": spearman_r,
                "Spearman_p": spearman_p
            })

# Convert to DataFrame
df_continuous_corr = pd.DataFrame(results_continuous).sort_values("Spearman_r", key=abs, ascending=False)
df_continuous_corr

,Variable,Pearson_r,Pearson_p,Spearman_r,Spearman_p
3,CO2_EMISS_CURR_PER_FLOOR_AREA,0.758721,1.028547e-26,0.724357,2.148101e-23
2,3_YR_ENERGY_SAVINGS_POTENTIAL,0.499120,6.214711e-10,0.551884,3.307038e-12
16,LOW_ENERGY_FIXED_LIGHT_COUNT,-0.244976,4.047253e-03,-0.431507,1.564378e-07
0,TOTAL_FLOOR_AREA,-0.193835,2.375234e-02,-0.358534,1.819363e-05
14,LIGHTING_COST_POTENTIAL,-0.253256,2.931434e-03,-0.317929,1.620865e-04
12,HOT_WATER_COST_POTENTIAL,-0.220864,9.768815e-03,-0.253496,2.903630e-03
9,HEATING_COST_CURRENT,0.275954,1.146667e-03,0.240834,4.737572e-03
6,GLAZED_AREA,0.114070,1.860616e-01,0.232430,6.470023e-03
1,3_YR_ENERGY_COST_CURRENT,0.273846,1.255411e-03,0.223833,8.804759e-03
5,WATER_HEATING_DEMAND,-0.042684,6.217240e-01,-0.139377,1.055974e-01


In [13]:
# ==========================
# Binary variables correlation with ENERGY_CONSUMPTION_CURRENT
# ==========================

from scipy.stats import pointbiserialr

binary_vars = [
    "MAINS_GAS_FLAG",
    "SOLAR_WATER_HEATING_FLAG",
    "FLAT_TOP_STOREY",
    "PHOTO_SUPPLY"
]

results_binary = []

for col in binary_vars:
    if col in df.columns:
        temp = df[[col, target]].dropna()
        # Try to convert to numeric (0/1)
        try:
            temp[col] = temp[col].astype(int)
        except ValueError:
            # If it can't be converted, skip this column
            print(f"Skipping {col}: not numeric or not binary")
            continue
        
        # Only compute correlation if there is more than one unique value
        if temp[col].nunique() > 1:
            r, p = pointbiserialr(temp[col], temp[target])
            results_binary.append({
                "Variable": col,
                "PointBiserial_r": r,
                "p_value": p
            })

df_binary_corr = pd.DataFrame(results_binary).sort_values("PointBiserial_r", key=abs, ascending=False)
df_binary_corr

Skipping PHOTO_SUPPLY: not numeric or not binary


,Variable,PointBiserial_r,p_value
1,FLAT_TOP_STOREY,-0.349859,0.356027
0,SOLAR_WATER_HEATING_FLAG,-0.065213,0.462798


In [14]:
# ==========================
# Categorical variables correlation with ENERGY_CONSUMPTION_CURRENT
# ==========================

from scipy.stats import f_oneway

categorical_vars = [
    "CONSTRUCTION_AGE_BAND",
    "PROPERTY_TYPE",
    "BUILT_FORM",
    "MAIN_HEATING_CATEGORY",
    "MAIN_FUEL",
    "MAIN_HEATING_CONTROLS",
    "MECHANICAL_VENTILATION",
    "ENERGY_TARIFF",
    "TENURE",
    "TRANSACTION_TYPE",
    "DATA_ZONE",
    "DATA_ZONE_2011",
    "LOCAL_AUTHORITY_LABEL",
    "CONSTITUENCY_LABEL"
]

results_categorical = []

for col in categorical_vars:
    if col in df.columns:
        temp = df[[col, target]].dropna()
        groups = [grp[target].values for name, grp in temp.groupby(col)]
        if len(groups) > 1:
            f_stat, p_val = f_oneway(*groups)
            # Eta-squared as effect size
            ss_between = sum(len(g)*(g.mean() - temp[target].mean())**2 for g in groups)
            eta_squared = ss_between / sum((temp[target] - temp[target].mean())**2)
            results_categorical.append({
                "Variable": col,
                "F_stat": f_stat,
                "p_value": p_val,
                "Eta_squared": eta_squared
            })

df_categorical_corr = pd.DataFrame(results_categorical).sort_values("Eta_squared", ascending=False)
df_categorical_corr

,Variable,F_stat,p_value,Eta_squared
5,MAIN_HEATING_CONTROLS,4.669151,1.507123e-08,0.513894
3,MAIN_HEATING_CATEGORY,15.074727,6.663024e-13,0.421772
9,TRANSACTION_TYPE,5.720342,9.002684e-06,0.238287
4,MAIN_FUEL,2.162229,1.781536e-02,0.180252
0,CONSTRUCTION_AGE_BAND,1.395074,1.981066e-01,0.096913
7,ENERGY_TARIFF,4.277772,6.529211e-03,0.093108
6,MECHANICAL_VENTILATION,6.403054,2.222566e-03,0.089051
8,TENURE,3.906019,1.035729e-02,0.081535
11,DATA_ZONE_2011,8.338991,4.525752e-03,0.058585
1,PROPERTY_TYPE,1.057938,3.693533e-01,0.023479


Here’s a structured analysis of your EPC correlation results, broken down by variable type and with actionable insights for archetyping in Fintry. I’ve also included recommendations for next steps.

---

## **1. Continuous Variables**

**Key results:**

| Variable                                                          | Pearson_r       | Pearson_p | Spearman_r | Spearman_p | Notes                                                                                             |
| ----------------------------------------------------------------- | --------------- | --------- | ---------- | ---------- | ------------------------------------------------------------------------------------------------- |
| CO2_EMISS_CURR_PER_FLOOR_AREA                                     | 0.759           | 1e-26     | 0.724      | 2e-23      | Very strong positive correlation with ENERGY_CONSUMPTION_CURRENT; top predictor                   |
| 3_YR_ENERGY_SAVINGS_POTENTIAL                                     | 0.499           | 6e-10     | 0.552      | 3e-12      | Moderate positive correlation; meaningful for retrofit potential                                  |
| HEATING_COST_CURRENT                                              | 0.276           | 0.001     | 0.241      | 0.005      | Smaller but significant correlation; expected as heating drives consumption                       |
| 3_YR_ENERGY_COST_CURRENT                                          | 0.274           | 0.001     | 0.224      | 0.009      | Similar to heating cost; reflective of overall energy load                                        |
| LIGHTING_COST_POTENTIAL                                           | -0.253          | 0.003     | -0.318     | 1.6e-4     | Negative correlation; could indicate small dwellings with efficient lighting have low consumption |
| LOW_ENERGY_FIXED_LIGHT_COUNT                                      | -0.245          | 0.004     | -0.432     | 1.5e-7     | Stronger Spearman correlation; may capture dwelling modernization                                 |
| TOTAL_FLOOR_AREA                                                  | -0.194          | 0.024     | -0.359     | 1.8e-5     | Negative Spearman correlation unexpected; likely due to small or atypical dwellings               |
| HOT_WATER_COST_POTENTIAL                                          | -0.221          | 0.01      | -0.254     | 0.003      | Weak to moderate negative correlation                                                             |
| GLAZED_AREA                                                       | 0.114           | 0.19      | 0.232      | 0.006      | Weak but possibly non-linear relation; could be captured in archetype visually                    |
| Other variables (SPACE_HEATING_DEMAND, NUMBER_HEATED_ROOMS, etc.) | low correlation |           |            |            | Not predictive individually                                                                       |

**Analysis:**

* **CO2_EMISS_CURR_PER_FLOOR_AREA** is by far the most predictive continuous variable, reflecting the building’s energy intensity independent of size.
* **3_YR_ENERGY_SAVINGS_POTENTIAL** is also meaningful for archetyping, especially for prioritizing retrofit pathways.
* **Other cost or area variables** show smaller but notable trends, suggesting that **floor area, glazing, and lighting modernization** could play secondary roles in differentiating archetypes.

**Takeaway for archetyping:** Use **CO2_EMISS_CURR_PER_FLOOR_AREA** as a primary continuous variable. Consider **3_YR_ENERGY_SAVINGS_POTENTIAL** or floor area if multiple variables are needed to distinguish subtypes.

---

## **2. Binary Variables**

| Variable                 | PointBiserial_r | p_value | Notes                                                                                        |
| ------------------------ | --------------- | ------- | -------------------------------------------------------------------------------------------- |
| FLAT_TOP_STOREY          | -0.350          | 0.356   | Moderate effect, but not statistically significant (likely low sample of flat-top dwellings) |
| SOLAR_WATER_HEATING_FLAG | -0.065          | 0.463   | Very weak; negligible for archetyping                                                        |

**Analysis:**

* Binary variables in your dataset are **not strongly correlated** with energy consumption.
* If included in archetyping, they could serve **as descriptive attributes** rather than predictors. For instance, “has solar water heating” could be noted for an archetype but **won’t define it**.

**Takeaway for archetyping:** Focus on **continuous and categorical variables**. Use binary features only as descriptive “labels” for archetypes.

---

## **3. Categorical Variables (ANOVA / Eta²)**

| Variable                                                      | F_stat     | p_value | Eta²                | Notes                                                                                  |
| ------------------------------------------------------------- | ---------- | ------- | ------------------- | -------------------------------------------------------------------------------------- |
| MAIN_HEATING_CONTROLS                                         | 4.67       | 1.5e-08 | 0.514               | Very strong effect; different heating control types significantly influence energy use |
| MAIN_HEATING_CATEGORY                                         | 15.07      | 6.7e-13 | 0.422               | Strongest categorical predictor; central for archetyping                               |
| TRANSACTION_TYPE                                              | 5.72       | 9e-06   | 0.238               | Moderate effect; may reflect occupancy/usage patterns                                  |
| MAIN_FUEL                                                     | 2.16       | 0.018   | 0.180               | Some effect; could differentiate gas vs electric-heated homes                          |
| CONSTRUCTION_AGE_BAND                                         | 1.40       | 0.198   | 0.097               | Surprisingly weak; may be confounded by retrofit work                                  |
| ENERGY_TARIFF, MECHANICAL_VENTILATION, TENURE, DATA_ZONE_2011 | 0.08–0.09  |         | Minor contributions |                                                                                        |
| BUILT_FORM, PROPERTY_TYPE, DATA_ZONE                          | 0.016–0.35 |         | Negligible          |                                                                                        |

**Analysis:**

* **MAIN_HEATING_CATEGORY** and **MAIN_HEATING_CONTROLS** are the **most statistically significant categorical variables**, explaining the largest portion of variance in energy consumption.
* **Transaction type** and **main fuel** offer secondary differentiation.
* Surprisingly, **construction age** has weaker statistical significance; this may indicate that **retrofits and upgrades have decoupled age from energy performance**.

**Takeaway for archetyping:** Use **heating system type and controls** as key categorical axes for archetypes. Secondary categories could include **fuel type** or **occupancy-related variables**.

---

## **4. Recommended Variables for Archetyping**

**Primary (strong predictors, statistically significant):**

1. **CO2_EMISS_CURR_PER_FLOOR_AREA** – continuous, energy intensity
2. **MAIN_HEATING_CATEGORY** – categorical, central to energy profile
3. **MAIN_HEATING_CONTROLS** – categorical, influences actual usage

**Secondary (descriptive / moderate predictors):**

* 3_YR_ENERGY_SAVINGS_POTENTIAL
* FLOOR_AREA or GLAZED_AREA (optional, for subtypes)
* Transaction type or fuel type
* Solar water heating / flat top storey flags (descriptive only)

---

## **5. Suggested Procedure for Archetyping Fintry EPC Data**

1. **Step 1 – Select variables:**

   * Continuous: `CO2_EMISS_CURR_PER_FLOOR_AREA` (+ optional secondary like 3_YR_ENERGY_SAVINGS_POTENTIAL)
   * Categorical: `MAIN_HEATING_CATEGORY`, `MAIN_HEATING_CONTROLS` (+ optional fuel type)

2. **Step 2 – Initial grouping:**

   * For categorical variables, define archetypes based on combinations (e.g., “Electric storage heaters + basic controls”).
   * For continuous variables, split into meaningful bins (e.g., tertiles of energy intensity) if necessary.

3. **Step 3 – Statistical verification:**

   * Compute **within-group variance** for continuous variables to ensure groups are homogenous.
   * Check **between-group variance** to confirm categories are well-separated.

4. **Step 4 – Alignment with local knowledge:**

   * Compare statistical archetypes with **existing dwelling types in Fintry** (Park Homes, Culcreuch, Menzies, Main Street, Dunmore).
   * Adjust archetypes for **intuitive understanding** for FDT while retaining statistical rigor.

5. **Step 5 – Use archetypes in modeling:**

   * Assign each home to a statistically-verified archetype.
   * Use a **representative dwelling per archetype** to run PHPP / hourly energy models.
   * Store results in the library of exemplar calculations for FDT.

---

✅ **Summary:**

* **Most predictive variables:** CO2_EMISS_CURR_PER_FLOOR_AREA, MAIN_HEATING_CATEGORY, MAIN_HEATING_CONTROLS
* **Next steps:** Create archetypes primarily along heating system and energy intensity, then verify statistical separation and alignment with known dwelling types.
* **Binary and minor categorical variables** can be used as descriptive attributes, but not to define archetypes.


